# 🛒 Global Superstore — Sales & Profit Analysis Notebook

**Objective:** Clean the dataset, compute KPIs, and visualize sales, profit, and segment performance.

> This notebook mirrors the logic in `dashboard.py`. Run cells top-to-bottom.
> Then launch the interactive app with: `streamlit run dashboard.py`

## Step 1 · Import Libraries

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)
print('Libraries loaded OK')

Libraries loaded OK


## Step 2 · Load & Inspect the Dataset

**Insight:** The file uses tab-separators (`\t`), not commas.
Using the wrong separator collapses all 27 columns into one unreadable column.
Always inspect the raw file with `head` before reading into pandas.

In [5]:
df = pd.read_csv(r"D:\Data Analyst\internship\Task 2\Global Superstore\Global Superstore.txt", sep="\t")

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Shape: (51290, 27)
Columns: ['Category', 'City', 'Country', 'Customer ID', 'Customer Name', 'Discount', 'Market', '记录数', 'Order Date', 'Order ID', 'Order Priority', 'Product ID', 'Product Name', 'Profit', 'Quantity', 'Region', 'Row ID', 'Sales', 'Segment', 'Ship Date', 'Ship Mode', 'Shipping Cost', 'State', 'Sub-Category', 'Year', 'Market2', 'weeknum']


,Category,City,Country,Customer ID,Customer Name,Discount,Market,记录数,Order Date,Order ID,Order Priority,Product ID,Product Name,Profit,Quantity,Region,Row ID,Sales,Segment,Ship Date,Ship Mode,Shipping Cost,State,Sub-Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.00,US,1,2011-01-07 00:00:00.000,CA-2011-130813,High,OFF-PA-10002005,Xerox 225,9.33,3,West,36624,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.00,US,1,2011-01-21 00:00:00.000,CA-2011-148614,Medium,OFF-PA-10002893,"Wirebound Service Call Books, 5 1/2"" x 4""",9.29,2,West,37033,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.00,US,1,2011-08-05 00:00:00.000,CA-2011-118962,Medium,OFF-PA-10000659,"Adams Phone Message Book, Professional, 400 Me...",9.84,3,West,31468,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32


## Step 3 · Clean the Data

**What we fix:**
1. Drop `记录数` (Chinese row-counter column) — no analytical value.
2. Parse `Order Date` strings → `datetime` — needed for time-series charts.
3. Coerce `Sales` and `Profit` to `float` — stray quotes can make pandas misread them as strings.
4. Verify no nulls in key columns — here the data is already complete, so no imputation needed.

In [6]:
# 1. Drop unnecessary column
df.drop(columns=['记录数'], inplace=True, errors='ignore')

# 2. Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'])

# 3. Ensure numeric types
df['Sales']  = pd.to_numeric(df['Sales'],  errors='coerce').fillna(0)
df['Profit'] = pd.to_numeric(df['Profit'], errors='coerce').fillna(0)

# 4. Null check
key_cols = ['Sales','Profit','Region','Category','Sub-Category','Segment','Customer Name']
print('Null counts in key columns:')
print(df[key_cols].isnull().sum())
print(f'\nCleaned dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')

Null counts in key columns:
Sales            0
Profit           0
Region           0
Category         0
Sub-Category     0
Segment          0
Customer Name    0
dtype: int64

Cleaned dataset: 51,290 rows, 26 columns


## Step 4 · Key Performance Indicators (KPIs)

**Insight:** These are the first numbers a stakeholder asks for.
`Profit Margin = Profit / Sales × 100` measures *efficiency* — two stores with the same
sales but different margins tell very different stories.

In [7]:
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
total_orders  = df['Order ID'].nunique()
profit_margin = total_profit / total_sales * 100

print(f'Total Sales    : ${total_sales:>12,.0f}')
print(f'Total Profit   : ${total_profit:>12,.0f}')
print(f'Unique Orders  : {total_orders:>13,}')
print(f'Profit Margin  : {profit_margin:>12.1f}%')

Total Sales    : $  12,642,905
Total Profit   : $   1,467,457
Unique Orders  :        25,035
Profit Margin  :         11.6%


## Step 5 · Sales by Region

**Insight:** Sorting bars by value removes the mental effort of visual comparison.
If a few regions dominate, the business should focus retention there;
smaller regions are growth opportunities.

In [8]:
region_sales = (
    df.groupby('Region')['Sales']
    .sum().reset_index()
    .sort_values('Sales', ascending=True)
)

fig = px.bar(
    region_sales, x='Sales', y='Region', orientation='h',
    color='Sales', color_continuous_scale='Blues',
    title='Total Sales by Region', text_auto='.2s'
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Step 6 · Profit Share by Category

**Insight:** With only 3 categories a donut chart works well — it shows *dominance* at a glance.
Technology typically yields higher margins than Furniture in global retail.

In [9]:
cat_profit = df.groupby('Category')['Profit'].sum().reset_index()

fig = px.pie(
    cat_profit, values='Profit', names='Category',
    title='Profit Share by Category',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    hole=0.35
)
fig.update_traces(textinfo='percent+label')
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Step 7 · Segment-wise Sales vs Profit

**Insight:** Plotting Sales and Profit side-by-side per segment reveals *efficiency gaps*.
A segment with high Sales but low Profit may be over-discounting or selling low-margin products.
That's an actionable pricing signal.

In [ ]:
seg = df.groupby('Segment')[['Sales','Profit']].sum().reset_index()
print(seg)

fig = px.bar(
    seg.melt(id_vars='Segment', value_vars=['Sales','Profit']),
    x='Segment', y='value', color='variable', barmode='group',
    title='Segment: Sales vs Profit',
    labels={'value': 'Amount ($)', 'variable': 'Metric'},
    color_discrete_sequence=['#4C72B0', '#DD8452']
)
fig.show()

## Step 8 · Top 5 Customers by Sales

**Insight:** Color-encoding Profit on a Sales bar chart immediately flags
*high-revenue but unprofitable* customers — a common problem when large
discounts are given to win volume orders.
Red bars = the customer costs you money despite buying a lot.

In [ ]:
top5 = (
    df.groupby('Customer Name')
    .agg(Total_Sales=('Sales','sum'), Total_Profit=('Profit','sum'))
    .reset_index()
    .sort_values('Total_Sales', ascending=False)
    .head(5)
)
print(top5.to_string(index=False))

fig = px.bar(
    top5.sort_values('Total_Sales'),
    x='Total_Sales', y='Customer Name', orientation='h',
    color='Total_Profit', color_continuous_scale='RdYlGn',
    title='Top 5 Customers by Sales (color = Profit)',
    labels={'Total_Sales': 'Total Sales ($)', 'Total_Profit': 'Profit ($)'},
    text_auto='.2s'
)
fig.show()

## Step 9 · Monthly Sales Trend

**Insight:** Monthly aggregation smooths daily noise while preserving seasonal patterns.
Look for recurring Q4 spikes — common in global retail — and whether there is
a positive growth slope across years.

In [ ]:
df['Month'] = df['Order Date'].dt.to_period('M').astype(str)
monthly = df.groupby('Month')['Sales'].sum().reset_index().sort_values('Month')

fig = px.line(
    monthly, x='Month', y='Sales',
    title='Monthly Sales Trend',
    labels={'Sales': 'Sales ($)', 'Month': ''},
    markers=True
)
fig.update_traces(line_color='#4C72B0', line_width=2)
fig.show()

## Step 10 · Bonus — Region × Sub-Category Profit Heatmap

**Insight:** Dark green cells = high-margin Region + Sub-Category combinations.
Red cells = loss-making combos that need a pricing or discount review.
This heatmap is too dense for a dashboard but perfect for a deep-dive analysis slide.

In [ ]:
import numpy as np

pivot = df.pivot_table(
    values='Profit', index='Region', columns='Sub-Category', aggfunc='sum'
)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
plt.colorbar(im, ax=ax, label='Profit ($)')
ax.set_title('Profit Heatmap: Region x Sub-Category', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

## Summary of Key Findings

| Finding | Business Implication |
|---|---|
| West & East dominate sales | Focus retention; upsell premium products |
| Technology has the highest profit margin | Prioritize tech promotions |
| Consumer segment leads in sales & profit | Primary audience for campaigns |
| Some top-revenue customers show low profit | Review per-account discount policy |
| Clear Q4 seasonal spikes | Stock and staff up in Q3 to capture demand |

---
**Next Step:** Run `streamlit run dashboard.py` for the interactive version.